**Phase 3d: Index Chunks in Azure AI Search**
- Synapse Notebook: 03_d_gold_index_chunks
- Azure AI Search kg-chunks index creation + upload.
- Trigger: After 03_c_gold_detect_communities
- Input:   silver_llm/not_personal/ on ADLS
- Output:  Azure AI Search index "kg-chunks"

**Enviroment variables setup**

In [ ]:
import os

os.environ["AZURE_OPENAI_ENDPOINT"] = "https://<YOUR_OPENAI_RESOURCE>.services.ai.azure.com/"
os.environ["AZURE_OPENAI_API_KEY"] = "<YOUR_OPENAI_KEY>"
os.environ["AZURE_OPENAI_DEPLOYMENT"] = "gpt-4o"
os.environ["AZURE_OPENAI_API_VERSION"] = "2025-01-01-preview"
os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"] = "text-embedding-3-small"
os.environ["ADLS_STORAGE_ACCOUNT"] = "<YOUR_ADLS_ACCOUNT>"
os.environ["ADLS_CONTAINER"] = "<YOUR_ADLS_CONTAINER>"
os.environ["ADLS_STORAGE_KEY"] = "<YOUR_ADLS_KEY>"
os.environ["COSMOS_GREMLIN_ENDPOINT"] = "wss://<YOUR_COSMOS_ACCOUNT>.gremlin.cosmos.azure.com:443/"
os.environ["COSMOS_GREMLIN_KEY"] = "<YOUR_COSMOS_KEY>"
os.environ["COSMOS_DATABASE"] = "email_cosmos_db"
os.environ["COSMOS_GRAPH"] = "email_cosmos_graph"
os.environ["COSMOS_NOSQL_ENDPOINT"] = ""
os.environ["COSMOS_NOSQL_KEY"] = ""
os.environ["AZURE_SEARCH_ENDPOINT"] = "https://<YOUR_SEARCH_SERVICE>.search.windows.net"
os.environ["AZURE_SEARCH_API_KEY"] = "<YOUR_SEARCH_KEY>"
os.environ["AZURE_SEARCH_INDEX_NAME"] = "kg-chunks"
os.environ["AZURE_SEARCH_ENTITY_INDEX_NAME"] = "kg-entities"
os.environ["AZURE_SEARCH_COMMUNITY_INDEX_NAME"] = "kg-communities"
os.environ["PIPELINE_MODE"] = "llm"
os.environ["PROCESS_ATTACHMENTS"] = "true"
print("Environment variables set.")

**Download code from ADLS**

In [ ]:
import os
import sys
from azure.identity import DefaultAzureCredential
from azure.storage.filedatalake import DataLakeServiceClient

service = DataLakeServiceClient(
    account_url=f"https://{os.environ['ADLS_STORAGE_ACCOUNT']}.dfs.core.windows.net",
    credential=os.environ["ADLS_STORAGE_KEY"]
)

fs = service.get_file_system_client(os.environ["ADLS_CONTAINER"])

for prefix in ["code/src", "code/config"]:
    for p in fs.get_paths(path=prefix, recursive=True):
        if not p.is_directory:
            local_path = p.name.replace("code/", "", 1)
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            with open(local_path, "wb") as f:
                file_client = fs.get_file_client(p.name)
                f.write(file_client.download_file().readall())

sys.path.insert(0, ".")
src_count = len(os.listdir("src")) if os.path.exists("src") else 0
config_count = len(os.listdir("config")) if os.path.exists("config") else 0
print(f"Downloaded: {src_count} items in src/, {config_count} items in config/")

**ADLS Adapter Init**

In [ ]:
import os
import sys
import json
import logging
from pathlib import Path
from datetime import datetime

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

sys.path.insert(0, ".")

from src.storage import ADLSAdapter

adapter = ADLSAdapter()

**Download Silver chunks from ADLS**

In [ ]:
SILVER_PATH = "silver_llm"
LOCAL_SILVER = "/tmp/pipeline/silver"

files = adapter.list_files(SILVER_PATH, "*.json")
print(f"Found {len(files)} Silver files on ADLS")

for f in files:
    rel_path = f.replace(f"{SILVER_PATH}/", "", 1)
    local_file = os.path.join(LOCAL_SILVER, rel_path)
    os.makedirs(os.path.dirname(local_file), exist_ok=True)
    with open(local_file, "wb") as fh:
        fh.write(adapter.read_bytes(f))

print(f"Downloaded Silver to {LOCAL_SILVER}")

**Check Blank Files.**

**Create index + embed + upload chunks**

In [ ]:
# --- Azure AI Search: Chunk index (dual vector + semantic config) ---
import json
from pathlib import Path
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SearchField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
    SemanticConfiguration, SemanticField, SemanticPrioritizedFields, SemanticSearch,
)
from azure.search.documents import SearchClient
from openai import AzureOpenAI

endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
api_key = os.environ["AZURE_SEARCH_API_KEY"]
credential = AzureKeyCredential(api_key)
index_name = os.environ.get("AZURE_SEARCH_INDEX_NAME", "kg-chunks")

index_client = SearchIndexClient(endpoint, credential)

try:
    index_client.delete_index(index_name)
    print(f"Deleted existing index '{index_name}'")
except Exception:
    pass

fields = [
    SearchField(name="chunk_id", type=SearchFieldDataType.String, key=True, filterable=True, searchable=True),
    SearchField(name="text_english", type=SearchFieldDataType.String, searchable=True),
    SearchField(name="summary", type=SearchFieldDataType.String, searchable=True),
    SearchField(name="text_english_vector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True, vector_search_dimensions=1536, vector_search_profile_name="hnsw-profile"),
    SearchField(name="summary_vector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True, vector_search_dimensions=1536, vector_search_profile_name="hnsw-profile"),
    SearchField(name="thread_id", type=SearchFieldDataType.String, filterable=True),
    SearchField(name="thread_subject", type=SearchFieldDataType.String, filterable=True, searchable=True),
    SearchField(name="source_type", type=SearchFieldDataType.String, filterable=True),
    SearchField(name="has_attachments", type=SearchFieldDataType.Boolean, filterable=True),
    SearchField(name="token_count", type=SearchFieldDataType.Int32, filterable=True),
    SearchField(name="email_sender", type=SearchFieldDataType.String, filterable=True, searchable=True),
    SearchField(name="sent_timestamp", type=SearchFieldDataType.String, filterable=True),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw-algo")],
    profiles=[VectorSearchProfile(name="hnsw-profile", algorithm_configuration_name="hnsw-algo")],
)

semantic_config = SemanticConfiguration(
    name="semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name="text_english")],
    ),
)
semantic_search = SemanticSearch(configurations=[semantic_config])

index = SearchIndex(name=index_name, fields=fields, vector_search=vector_search, semantic_search=semantic_search)
index_client.create_or_update_index(index)
print(f"Index '{index_name}' created (dual vector + semantic config)")

# Embedding client
oai_client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)
emb_deployment = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")
EMBED_BATCH_SIZE = 16

def embed_batch(texts):
    truncated = [t[:8000] for t in texts]
    response = oai_client.embeddings.create(input=truncated, model=emb_deployment)
    return [d.embedding for d in response.data]

# Load chunks from Silver
search_client = SearchClient(endpoint, index_name, credential)
silver_base = "/tmp/pipeline/silver"
chunks_to_process = []
skipped = 0

for pattern in ["not_personal/email_chunks", "not_personal/attachment_chunks", "not_personal/document_chunks"]:
    chunk_dir = Path(f"{silver_base}/{pattern}")
    if not chunk_dir.exists():
        continue
    for chunk_file in chunk_dir.glob("*.json"):
        try:
            with open(chunk_file, 'r') as f:
                chunk = json.load(f)
        except (json.JSONDecodeError, Exception):
            skipped += 1
            continue

        text_english = chunk.get("text_english") or chunk.get("text_anonymized", "")
        if not text_english:
            skipped += 1
            continue

        chunks_to_process.append(chunk)

print(f"Loaded {len(chunks_to_process)} chunks ({skipped} skipped), embedding in batches of {EMBED_BATCH_SIZE}...")

# Batch embed + stream upload
documents = []
total_ok = 0

for i in range(0, len(chunks_to_process), EMBED_BATCH_SIZE):
    batch = chunks_to_process[i:i + EMBED_BATCH_SIZE]

    text_inputs = []
    summary_inputs = []
    for chunk in batch:
        subject = chunk.get("thread_subject", "")
        prefix = f"Subject: {subject}. " if subject else ""
        text_english = chunk.get("text_english") or chunk.get("text_anonymized", "")
        summary = chunk.get("summary", "")
        text_inputs.append(f"{prefix}{text_english}")
        summary_inputs.append(f"{prefix}{summary}" if summary and summary != text_english else None)

    text_vectors = embed_batch(text_inputs)

    unique_summaries = [s for s in summary_inputs if s]
    summary_vectors = embed_batch(unique_summaries) if unique_summaries else []
    sv_iter = iter(summary_vectors)

    for j, chunk in enumerate(batch):
        documents.append({
            "chunk_id": chunk.get("chunk_id", ""),
            "text_english": chunk.get("text_english") or chunk.get("text_anonymized", ""),
            "summary": chunk.get("summary", ""),
            "text_english_vector": text_vectors[j],
            "summary_vector": next(sv_iter) if summary_inputs[j] else text_vectors[j],
            "thread_id": chunk.get("thread_id", ""),
            "thread_subject": chunk.get("thread_subject", ""),
            "source_type": chunk.get("source_type", "email"),
            "has_attachments": chunk.get("has_attachments", False),
            "token_count": chunk.get("token_count", 0),
            "email_sender": chunk.get("email_sender", ""),
            "sent_timestamp": chunk.get("sent_timestamp", ""),
        })

    while len(documents) >= 100:
        result = search_client.upload_documents(documents[:100])
        total_ok += sum(1 for r in result if r.succeeded)
        documents = documents[100:]

    if (i // EMBED_BATCH_SIZE) % 50 == 0:
        print(f"  Progress: {min(i + EMBED_BATCH_SIZE, len(chunks_to_process))}/{len(chunks_to_process)} embedded,{total_ok} indexed")

if documents:
    result = search_client.upload_documents(documents)
    total_ok += sum(1 for r in result if r.succeeded)

print(f"Done — {total_ok} chunks indexed (dual vector + semantic)")